# Text Generation using GPT-2

Welcome to the creative world of AI text generation! This notebook demonstrates how to use GPT-2 for various text generation tasks.

## 🎯 **What You'll Learn:**
- **Text Continuation**: Complete prompts with AI-generated text
- **Creative Writing**: Generate stories, poems, dialogues, and essays
- **Story Continuation**: Create multiple endings for story beginnings
- **Parameter Tuning**: Control creativity, randomness, and style
- **Interactive Generation**: Real-time text generation playground

## 🤖 **About GPT-2:**
GPT-2 (Generative Pre-trained Transformer 2) is an autoregressive language model that:
- Predicts the next word based on previous context
- Can generate human-like text across various domains
- Supports different generation strategies (greedy, sampling, beam search)
- Available in multiple sizes: small (117M), medium (345M), large (762M), xl (1.5B)

## 🎨 **Creative Applications:**
- ✅ Story writing and continuation
- ✅ Poetry generation
- ✅ Dialogue creation
- ✅ Essay writing assistance
- ✅ Content brainstorming
- ✅ Creative prompts exploration

In [1]:
# Import Required Libraries
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer, pipeline, set_seed
from typing import List, Dict, Union, Optional
import random
import re
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducible results
set_seed(42)

# Check package versions and GPU availability
print("📦 Package Information:")
print(f"PyTorch version: {torch.__version__}")
print(f"Device available: {'GPU' if torch.cuda.is_available() else 'CPU'}")

try:
    import transformers
    print(f"Transformers version: {transformers.__version__}")
    print("✅ All required packages are available!")
except ImportError as e:
    print(f"❌ Missing package: {e}")
    print("Install with: pip install transformers torch")

print(f"\n🎲 Random seed set to 42 for reproducible results")


import requests
from huggingface_hub import configure_http_backend

def backend_factory() -> requests.Session:
    session = requests.Session()
    session.verify = False
    return session

configure_http_backend(backend_factory=backend_factory)

c:\Users\AnithaMudigoudar\Downloads\Workshop_Transformers\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


📦 Package Information:
PyTorch version: 2.9.0+cpu
Device available: CPU
Transformers version: 4.57.1
✅ All required packages are available!

🎲 Random seed set to 42 for reproducible results


## 🤖 GPT2TextGenerator Class

Let's build our text generation system with modular components. Each method will handle different aspects of text generation.

In [2]:
# GPT2TextGenerator Class - Initialization
class GPT2TextGenerator:
    """
    A text generation class using GPT-2 model.
    
    Supported model sizes:
    - small: gpt2 (117M parameters) - Fast, lower quality
    - medium: gpt2-medium (345M parameters) - Balanced
    - large: gpt2-large (762M parameters) - High quality, slower
    - xl: gpt2-xl (1.5B parameters) - Highest quality, very slow
    """
    
    def __init__(self, model_size: str = "small"):
        """
        Initialize the GPT-2 text generator.
        
        Args:
            model_size (str): Model size ("small", "medium", "large", "xl")
                             Note: Larger models require more memory and time
        """
        self.model_size = model_size
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        
        print(f"🔧 Initializing GPT2TextGenerator")
        print(f"📱 Using device: {self.device}")
        print(f"📏 Model size: {self.model_size}")
        
        self.tokenizer = None
        self.model = None
        self.generator = None
        self.is_loaded = False

print("✅ GPT2TextGenerator class defined!")

✅ GPT2TextGenerator class defined!


In [3]:
# GPT2TextGenerator - Model Loading
def load_model(self):
    """Load the GPT-2 model and tokenizer."""
    try:
        # Determine model name based on size
        model_mapping = {
            "small": "gpt2",
            "medium": "gpt2-medium", 
            "large": "gpt2-large",
            "xl": "gpt2-xl"
        }
        
        model_name = model_mapping.get(self.model_size, "gpt2")
        
        print(f"📥 Loading GPT-2 model: {model_name}")
        print(f"⏳ This may take a few moments...")
        
        self.tokenizer = GPT2Tokenizer.from_pretrained(model_name)
        self.model = GPT2LMHeadModel.from_pretrained(model_name)
        
        # Set pad token (GPT-2 doesn't have one by default)
        self.tokenizer.pad_token = self.tokenizer.eos_token
        
        self.model.to(self.device)
        
        # Create pipeline for easier text generation
        self.generator = pipeline(
            "text-generation",
            model=self.model,
            tokenizer=self.tokenizer,
            device=0 if self.device.type == "cuda" else -1
        )
        
        self.is_loaded = True
        print("✅ Model loaded successfully!")
        
        # Display model info
        num_params = sum(p.numel() for p in self.model.parameters()) / 1e6
        print(f"📊 Model parameters: ~{num_params:.0f}M")
        print(f"💾 Model name: {model_name}")
        
    except Exception as e:
        print(f"❌ Error loading model: {e}")
        print("💡 Try using a smaller model (e.g., 'small') or check your internet connection")
        self.is_loaded = False

# Add method to class
GPT2TextGenerator.load_model = load_model

print("✅ Model loading method added!")

✅ Model loading method added!


In [4]:
# GPT2TextGenerator - Basic Text Generation
def generate_text(
    self,
    prompt: str,
    max_length: int = 100,
    num_return_sequences: int = 1,
    temperature: float = 0.8,
    do_sample: bool = True,
    top_k: int = 50,
    top_p: float = 0.95,
    repetition_penalty: float = 1.1
) -> List[Dict[str, Union[str, float]]]:
    """
    Generate text based on a prompt with customizable parameters.
    
    Args:
        prompt (str): Input text to continue
        max_length (int): Maximum length of generated text
        num_return_sequences (int): Number of sequences to generate
        temperature (float): Sampling temperature (0.0 = deterministic, 1.0 = random)
        do_sample (bool): Whether to use sampling
        top_k (int): Top-k sampling parameter
        top_p (float): Top-p (nucleus) sampling parameter
        repetition_penalty (float): Penalty for repetition
        
    Returns:
        List[dict]: Generated text sequences with metadata
    """
    if not self.is_loaded:
        return [{"error": "Model not loaded"}]
    
    try:
        results = self.generator(
            prompt,
            max_length=max_length,
            num_return_sequences=num_return_sequences,
            temperature=temperature,
            do_sample=do_sample,
            top_k=top_k,
            top_p=top_p,
            repetition_penalty=repetition_penalty,
            pad_token_id=self.tokenizer.eos_token_id,
            return_full_text=True
        )
        
        formatted_results = []
        for i, result in enumerate(results):
            generated_text = result['generated_text']
            # Remove the prompt from the generated text to show only new content
            new_text = generated_text[len(prompt):].strip()
            
            formatted_results.append({
                "sequence_id": i + 1,
                "full_text": generated_text,
                "generated_part": new_text,
                "prompt": prompt,
                "length": len(generated_text.split()),
                "generated_length": len(new_text.split()),
                "parameters": {
                    "temperature": temperature,
                    "top_k": top_k,
                    "top_p": top_p
                }
            })
        
        return formatted_results
        
    except Exception as e:
        print(f"❌ Error generating text: {e}")
        return [{"error": str(e)}]

# Add method to class
GPT2TextGenerator.generate_text = generate_text

print("✅ Basic text generation method added!")

✅ Basic text generation method added!


In [5]:
# GPT2TextGenerator - Creative Writing
def creative_writing(
    self,
    prompt: str,
    style: str = "story",
    length: str = "medium"
) -> Dict[str, str]:
    """
    Generate creative writing based on style and length preferences.
    
    Args:
        prompt (str): Writing prompt
        style (str): Writing style ("story", "poem", "dialogue", "essay")
        length (str): Desired length ("short", "medium", "long")
        
    Returns:
        dict: Generated creative writing with metadata
    """
    if not self.is_loaded:
        return {"error": "Model not loaded"}
    
    # Length settings
    length_mapping = {
        "short": 80,
        "medium": 150,
        "long": 250
    }
    
    max_length = length_mapping.get(length, 150)
    
    # Style-specific parameters
    style_params = {
        "poem": {"temperature": 0.9, "top_p": 0.9},
        "dialogue": {"temperature": 0.8, "top_p": 0.85},
        "essay": {"temperature": 0.7, "top_p": 0.9},
        "story": {"temperature": 0.8, "top_p": 0.95}
    }
    
    params = style_params.get(style, style_params["story"])
    
    # Style-specific prompt formatting
    style_prompts = {
        "poem": f"Write a poem about {prompt}:\n\n",
        "dialogue": f"Create a dialogue about {prompt}:\n\n",
        "essay": f"Write an essay about {prompt}:\n\n",
        "story": f"Once upon a time, {prompt}"
    }
    
    formatted_prompt = style_prompts.get(style, prompt)
    
    result = self.generate_text(
        formatted_prompt,
        max_length=max_length,
        temperature=params["temperature"],
        top_p=params["top_p"],
        num_return_sequences=1
    )[0]
    
    if "error" in result:
        return result
    
    return {
        "style": style,
        "length": length,
        "prompt": prompt,
        "formatted_prompt": formatted_prompt,
        "generated_text": result.get("full_text", ""),
        "generated_part": result.get("generated_part", ""),
        "word_count": result.get("length", 0),
        "parameters_used": result.get("parameters", {})
    }

# Add method to class
GPT2TextGenerator.creative_writing = creative_writing

print("✅ Creative writing method added!")

✅ Creative writing method added!


In [6]:
# GPT2TextGenerator - Story Continuation
def continue_story(self, story_beginning: str, num_continuations: int = 3) -> List[Dict[str, str]]:
    """
    Generate multiple continuations for a story beginning.
    
    Args:
        story_beginning (str): Beginning of the story
        num_continuations (int): Number of different continuations
        
    Returns:
        List[dict]: Different story continuations
    """
    if not self.is_loaded:
        return [{"error": "Model not loaded"}]
    
    results = self.generate_text(
        story_beginning,
        max_length=200,
        num_return_sequences=num_continuations,
        temperature=0.85,
        top_p=0.9
    )
    
    continuations = []
    for i, result in enumerate(results):
        if "error" not in result:
            continuations.append({
                "continuation_id": i + 1,
                "story_beginning": story_beginning,
                "continuation": result["generated_part"],
                "full_story": result["full_text"],
                "word_count": result["generated_length"]
            })
    
    return continuations

# Add method to class
GPT2TextGenerator.continue_story = continue_story

print("✅ Story continuation method added!")

✅ Story continuation method added!


## 🚀 Initialize GPT-2 Generator

Let's create our text generator instance. You can modify the model size here - start with "small" for faster loading and testing.

In [9]:
# Initialize GPT-2 Text Generator
print("🔧 Creating GPT-2 Text Generator...")

# Model size options:
# - "small": Fast, lower quality (117M parameters)
# - "medium": Balanced (345M parameters) 
# - "large": High quality, slower (762M parameters)
# - "xl": Highest quality, very slow (1.5B parameters)

generator = GPT2TextGenerator(model_size="small")

# Load the model
print("\n📥 Loading GPT-2 model...")
generator.load_model()

if generator.is_loaded:
    print(f"\n🎉 GPT-2 Generator ready!")
    print(f"🤖 Model: {generator.model_size}")
    print(f"📱 Device: {generator.device}")
else:
    print("\n⚠️ Generator failed to load. Check the error messages above.")

🔧 Creating GPT-2 Text Generator...
🔧 Initializing GPT2TextGenerator
📱 Using device: cpu
📏 Model size: small

📥 Loading GPT-2 model...
📥 Loading GPT-2 model: gpt2
⏳ This may take a few moments...


'('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None))' thrown while requesting HEAD https://huggingface.co/gpt2/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].
Device set to use cpu


✅ Model loaded successfully!
📊 Model parameters: ~124M
💾 Model name: gpt2

🎉 GPT-2 Generator ready!
🤖 Model: small
📱 Device: cpu


## 📝 Basic Text Generation

Let's start with simple text completion. GPT-2 will continue your prompts based on patterns it learned during training.

In [10]:
# Basic Text Generation Examples
test_prompts = [
    "The future of artificial intelligence is",
    "In a world where robots and humans coexist",
    "The most important discovery in science was",
    "Climate change can be addressed by",
    "The secret to happiness is"
]

print("🧪 Basic Text Generation Results:")
print("=" * 50)

for i, prompt in enumerate(test_prompts, 1):
    print(f"\n{i}. 💭 Prompt: '{prompt}'")
    print("-" * 40)
    
    # Generate text with moderate creativity
    results = generator.generate_text(
        prompt, 
        max_length=80, 
        temperature=0.8,
        num_return_sequences=1
    )
    
    for result in results:
        if "error" not in result:
            print(f"🤖 Generated: {result['generated_part']}")
            print(f"📊 Words added: {result['generated_length']}")
        else:
            print(f"❌ Error: {result['error']}")

print(f"\n✅ Completed {len(test_prompts)} text generations!")

Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.


Both `max_new_tokens` (=256) and `max_length`(=80) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🧪 Basic Text Generation Results:

1. 💭 Prompt: 'The future of artificial intelligence is'
----------------------------------------


Both `max_new_tokens` (=256) and `max_length`(=80) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🤖 Generated: uncertain and could be even more important than the near-term outcome. In a new paper titled "Neural Networks: A Review", researchers from Harvard University's Watson Institute provide some insight into these issues at how they may shape AI systems as we know them, including their potential impact on our daily lives.
"These studies show that machine learning models can help us to understand complex patterns in human behavior," says Peter Hochberg, an assistant professor of computer science who led the research team working on the study. "With this kind (of neural network) you have very high accuracy or precision with which one will respond." He adds he hopes such machines would eventually become self sufficient – meaning humans won't need brain processing power until technology improves so much about it makes sense for those still struggling to adapt quickly to modern algorithms. The scientists suggest robots might ultimately drive many tasks better then conventional compu

Both `max_new_tokens` (=256) and `max_length`(=80) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🤖 Generated: , what happens when one finds out that they're the enemy of humanity? And why is it so easy to believe in something you don't understand?"


 (Photo by Brian Hall. All rights reserved.)
📊 Words added: 33

3. 💭 Prompt: 'The most important discovery in science was'
----------------------------------------


KeyboardInterrupt: 

## 🎨 Creative Writing

Now let's explore creative writing! GPT-2 can adapt its style based on the type of content you want to generate.

In [ ]:
# Creative Writing Examples
creative_prompts = [
    ("a mysterious forest", "story"),
    ("the sound of rain", "poem"),
    ("technology and humanity", "dialogue"),
    ("the importance of education", "essay")
]

print("🎨 Creative Writing Results:")
print("=" * 40)

for i, (prompt, style) in enumerate(creative_prompts, 1):
    print(f"\n{i}. 🎭 {style.title()} about '{prompt}'")
    print("-" * 50)
    
    result = generator.creative_writing(prompt, style, "medium")
    
    if "error" not in result:
        # Display the generated content
        if style == "story":
            content = result["generated_part"]
        else:
            # For other styles, show the full formatted text
            content = result["generated_text"]
            # Remove the formatted prompt if it's still there
            if result["formatted_prompt"] in content:
                content = content.replace(result["formatted_prompt"], "").strip()
        
        print(f"✨ Generated {style}:")
        print(content[:300] + "..." if len(content) > 300 else content)
        print(f"\n📊 Word count: {result['word_count']}")
        print(f"🎛️ Temperature: {result['parameters_used'].get('temperature', 'N/A')}")
    else:
        print(f"❌ Error: {result['error']}")

print(f"\n✅ Generated {len(creative_prompts)} creative pieces!")

## 📖 Story Continuation

One of GPT-2's most interesting capabilities is generating multiple possible continuations for the same story beginning.

In [ ]:
# Story Continuation Examples
story_beginnings = [
    "Sarah opened the old chest in the attic and found",
    "The spaceship landed in the middle of the city, and",
    "After 100 years of sleep, the ancient dragon awoke to discover"
]

for story_start in story_beginnings:
    print(f"📖 Story Beginning: '{story_start}'")
    print("=" * 60)
    
    continuations = generator.continue_story(story_start, num_continuations=3)
    
    for cont in continuations:
        if "error" not in cont:
            print(f"\n🔮 Continuation {cont['continuation_id']}:")
            print("-" * 30)
            # Show a reasonable length of continuation
            continuation_text = cont["continuation"]
            display_text = continuation_text[:200] + "..." if len(continuation_text) > 200 else continuation_text
            print(display_text)
            print(f"📊 Words: {cont['word_count']}")
        else:
            print(f"❌ Error in continuation: {cont.get('error', 'Unknown error')}")
    
    print("\n" + "🌟" * 50 + "\n")

print("✅ Story continuation demo completed!")

## 🎛️ Parameter Playground

Understanding how different parameters affect text generation is crucial for getting the results you want. Let's experiment!

In [ ]:
# Temperature Experiment
test_prompt = "The secret to happiness is"
print(f"🧪 Testing Temperature Effects with: '{test_prompt}'")
print("=" * 60)

temperatures = [0.3, 0.7, 1.0, 1.3]

for temp in temperatures:
    print(f"\n🌡️ Temperature: {temp}")
    if temp <= 0.5:
        print("   (Conservative, predictable text)")
    elif temp <= 1.0:
        print("   (Balanced creativity)")
    else:
        print("   (High creativity, more random)")
    
    print("-" * 30)
    
    result = generator.generate_text(
        test_prompt,
        max_length=60,
        temperature=temp,
        num_return_sequences=1,
        do_sample=True
    )[0]
    
    if "error" not in result:
        print(f"Generated: {result['generated_part']}")
    else:
        print(f"Error: {result['error']}")

print(f"\n💡 Key Insight: Lower temperature = more focused, higher temperature = more creative!")

In [ ]:
# Sampling Strategy Comparison
print(f"🎯 Comparing Sampling Strategies with: '{test_prompt}'")
print("=" * 60)

strategies = [
    {"name": "Greedy Decoding", "do_sample": False, "description": "Always picks most likely word"},
    {"name": "Top-k (k=10)", "do_sample": True, "top_k": 10, "top_p": 1.0, "description": "Choose from top 10 most likely words"},
    {"name": "Top-p (p=0.8)", "do_sample": True, "top_k": 0, "top_p": 0.8, "description": "Choose from words that sum to 80% probability"},
    {"name": "Combined (k=40, p=0.9)", "do_sample": True, "top_k": 40, "top_p": 0.9, "description": "Combined top-k and top-p sampling"}
]

for strategy in strategies:
    print(f"\n🔧 {strategy['name']}")
    print(f"   {strategy['description']}")
    print("-" * 40)
    
    # Extract parameters
    params = {k: v for k, v in strategy.items() if k not in ["name", "description"]}
    
    result = generator.generate_text(
        test_prompt, 
        max_length=60, 
        temperature=0.8,
        **params
    )[0]
    
    if "error" not in result:
        print(f"Result: {result['generated_part']}")
    else:
        print(f"Error: {result['error']}")

print(f"\n💡 Experiment with different strategies to find your preferred style!")

## 🎮 Interactive Text Generation

**Your Turn!** Use this cell to experiment with your own prompts and parameters. Modify the variables below and run the cell to see different results.